# 02 — Ingest Raw Source Data

## Banking Reporting Platform

This notebook loads the four source CSV extracts into PostgreSQL's `raw` schema.

The purpose of the raw layer is:

> **Preserve the source data as received before applying business-rule validation or cleaning.**

The intentionally bad source records are loaded too. They are not rejected here.

```text
CSV source files
        ↓
      raw
        ↓
    staging
        ↓
      marts
```

For this portfolio project, each CSV represents the latest full source extract. The notebook therefore performs a **full-refresh raw load**: create the raw tables if required, truncate them, load every source row, add ingestion metadata, and reconcile source counts to database counts.


## 1. Imports and Project Paths

The source files are read with Python's built-in `csv` module so dates, amounts, and codes are not converted before they reach the raw layer.


In [1]:
from pathlib import Path
import csv

import pandas as pd
import psycopg
from psycopg import sql
from IPython.display import display


def find_project_root(start: Path) -> Path:
    start = start.resolve()

    for candidate in [start, *start.parents]:
        if (candidate / "data" / "raw").exists():
            return candidate

    raise FileNotFoundError(
        "Could not locate the project root. Expected a data/raw directory "
        "in the current directory or one of its parents."
    )


PROJECT_ROOT = find_project_root(Path.cwd())
RAW_DIR = PROJECT_ROOT / "data" / "raw"
ENV_PATH = PROJECT_ROOT / ".env"

print(f"Project root: {PROJECT_ROOT}")
print(f"Raw source directory: {RAW_DIR}")
print(f"Environment file: {ENV_PATH}")


Project root: C:\Users\Admin\Projects\banking-reporting-platform
Raw source directory: C:\Users\Admin\Projects\banking-reporting-platform\data\raw
Environment file: C:\Users\Admin\Projects\banking-reporting-platform\.env


## 2. Read the Local Database Configuration

The notebook reads the existing `.env` file directly. The password is never printed.


In [2]:
def read_env_file(path: Path) -> dict[str, str]:
    if not path.exists():
        raise FileNotFoundError(
            f"{path} was not found. Create .env from .env.example before continuing."
        )

    values = {}

    for raw_line in path.read_text(encoding="utf-8").splitlines():
        line = raw_line.strip()

        if not line or line.startswith("#") or "=" not in line:
            continue

        key, value = line.split("=", 1)
        values[key.strip()] = value.strip().strip('"').strip("'")

    return values


env = read_env_file(ENV_PATH)

required_env_keys = {
    "POSTGRES_USER",
    "POSTGRES_PASSWORD",
    "POSTGRES_DB",
    "POSTGRES_PORT",
}

missing_env_keys = sorted(required_env_keys - env.keys())

if missing_env_keys:
    raise KeyError(f"Missing required .env values: {missing_env_keys}")


DB_CONFIG = {
    "host": "localhost",
    "port": int(env["POSTGRES_PORT"]),
    "dbname": env["POSTGRES_DB"],
    "user": env["POSTGRES_USER"],
    "password": env["POSTGRES_PASSWORD"],
}

print(
    "PostgreSQL target:",
    f"{DB_CONFIG['user']}@{DB_CONFIG['host']}:{DB_CONFIG['port']}/{DB_CONFIG['dbname']}",
)


PostgreSQL target: banking_admin@localhost:5432/analytics


## 3. Define the Raw Source Contract

All source business fields are stored as PostgreSQL `TEXT` in raw. Typed conversion happens in staging.


In [3]:
RAW_SOURCES = {
    "customers": {
        "file": RAW_DIR / "customers.csv",
        "columns": [
            "customer_id",
            "customer_since_date",
            "customer_status",
        ],
    },
    "accounts": {
        "file": RAW_DIR / "accounts.csv",
        "columns": [
            "account_id",
            "account_type",
            "account_status",
            "opened_date",
            "closed_date",
        ],
    },
    "customer_accounts": {
        "file": RAW_DIR / "customer_accounts.csv",
        "columns": [
            "customer_id",
            "account_id",
            "holder_role",
        ],
    },
    "transactions": {
        "file": RAW_DIR / "transactions.csv",
        "columns": [
            "transaction_id",
            "account_id",
            "transaction_timestamp",
            "transaction_type",
            "channel_code",
            "amount",
            "currency_code",
            "status",
        ],
    },
}

for table_name, spec in RAW_SOURCES.items():
    print(f"{table_name:20s} <- {spec['file'].name}")


customers            <- customers.csv
accounts             <- accounts.csv
customer_accounts    <- customer_accounts.csv
transactions         <- transactions.csv


## 4. Validate the Minimum Ingestion Contract

The detailed profiling already happened in notebook 01. Here we only ensure the expected file exists and has the expected header before loading it.


In [4]:
def inspect_csv(path: Path, expected_columns: list[str]) -> dict:
    if not path.exists():
        raise FileNotFoundError(f"Missing source file: {path}")

    with path.open("r", newline="", encoding="utf-8") as handle:
        reader = csv.DictReader(handle)
        actual_columns = reader.fieldnames or []

        if actual_columns != expected_columns:
            raise ValueError(
                f"{path.name} has unexpected columns.\n"
                f"Expected: {expected_columns}\n"
                f"Actual:   {actual_columns}"
            )

        row_count = sum(1 for _ in reader)

    return {
        "file": path.name,
        "rows": row_count,
        "columns": len(actual_columns),
        "schema_matches": True,
    }


source_profile = pd.DataFrame(
    [
        inspect_csv(spec["file"], spec["columns"])
        for spec in RAW_SOURCES.values()
    ]
)

display(source_profile)


,file,rows,columns,schema_matches
0,customers.csv,1003,3,True
1,accounts.csv,1253,5,True
2,customer_accounts.csv,1355,3,True
3,transactions.csv,50010,8,True


## 5. Test the PostgreSQL Connection

Docker Desktop and PostgreSQL must be running first:

```powershell
docker compose up -d
```

The notebook connects from Windows to PostgreSQL through `localhost`.


In [5]:
def get_connection():
    return psycopg.connect(**DB_CONFIG)


with get_connection() as conn:
    with conn.cursor() as cur:
        cur.execute(
            '''
            SELECT
                current_database(),
                current_user,
                version();
            '''
        )
        database_name, database_user, postgres_version = cur.fetchone()

print(f"Connected database: {database_name}")
print(f"Connected user:     {database_user}")
print(f"PostgreSQL:         {postgres_version.split(',')[0]}")


Connected database: analytics
Connected user:     banking_admin
PostgreSQL:         PostgreSQL 17.11 (Debian 17.11-1.pgdg13+2) on x86_64-pc-linux-gnu


## 6. Create the Raw Schema and Tables

The definitions match `04_physical_data_model.md`.

Each table includes `_source_file` and `_ingested_at`.


In [6]:
RAW_DDL = '''
CREATE SCHEMA IF NOT EXISTS raw;

CREATE TABLE IF NOT EXISTS raw.customers (
    customer_id TEXT,
    customer_since_date TEXT,
    customer_status TEXT,
    _ingested_at TIMESTAMPTZ NOT NULL DEFAULT now(),
    _source_file TEXT NOT NULL
);

CREATE TABLE IF NOT EXISTS raw.accounts (
    account_id TEXT,
    account_type TEXT,
    account_status TEXT,
    opened_date TEXT,
    closed_date TEXT,
    _ingested_at TIMESTAMPTZ NOT NULL DEFAULT now(),
    _source_file TEXT NOT NULL
);

CREATE TABLE IF NOT EXISTS raw.customer_accounts (
    customer_id TEXT,
    account_id TEXT,
    holder_role TEXT,
    _ingested_at TIMESTAMPTZ NOT NULL DEFAULT now(),
    _source_file TEXT NOT NULL
);

CREATE TABLE IF NOT EXISTS raw.transactions (
    transaction_id TEXT,
    account_id TEXT,
    transaction_timestamp TEXT,
    transaction_type TEXT,
    channel_code TEXT,
    amount TEXT,
    currency_code TEXT,
    status TEXT,
    _ingested_at TIMESTAMPTZ NOT NULL DEFAULT now(),
    _source_file TEXT NOT NULL
);
'''


with get_connection() as conn:
    with conn.cursor() as cur:
        cur.execute(RAW_DDL)

print("Raw schema and tables are ready.")


Raw schema and tables are ready.


## 7. Full-Refresh the Raw Layer

These files represent complete source extracts, so the current raw rows are removed before the new load. No source record is filtered here.


In [7]:
RAW_TABLES = [
    "customers",
    "accounts",
    "customer_accounts",
    "transactions",
]

with get_connection() as conn:
    with conn.cursor() as cur:
        cur.execute(
            '''
            TRUNCATE TABLE
                raw.transactions,
                raw.customer_accounts,
                raw.accounts,
                raw.customers;
            '''
        )

print("Existing raw rows removed.")


Existing raw rows removed.


## 8. Load the CSV Files with PostgreSQL COPY

`COPY` is PostgreSQL's high-throughput loading mechanism.

For each row, the notebook writes the original source columns plus `_source_file`. PostgreSQL automatically populates `_ingested_at`.


In [8]:
def copy_csv_to_raw(
    table_name: str,
    csv_path: Path,
    source_columns: list[str],
) -> int:
    target_columns = [*source_columns, "_source_file"]

    copy_statement = sql.SQL("COPY raw.{} ({}) FROM STDIN").format(
        sql.Identifier(table_name),
        sql.SQL(", ").join(sql.Identifier(column) for column in target_columns),
    )

    loaded_rows = 0

    with csv_path.open("r", newline="", encoding="utf-8") as handle:
        reader = csv.DictReader(handle)

        with get_connection() as conn:
            with conn.cursor() as cur:
                with cur.copy(copy_statement) as copy:
                    for row in reader:
                        values = [row[column] for column in source_columns]
                        values.append(csv_path.name)

                        copy.write_row(values)
                        loaded_rows += 1

    return loaded_rows


load_results = []

for table_name, spec in RAW_SOURCES.items():
    rows_loaded = copy_csv_to_raw(
        table_name=table_name,
        csv_path=spec["file"],
        source_columns=spec["columns"],
    )

    load_results.append(
        {
            "table": f"raw.{table_name}",
            "source_file": spec["file"].name,
            "rows_loaded": rows_loaded,
        }
    )

load_results = pd.DataFrame(load_results)
display(load_results)


,table,source_file,rows_loaded
0,raw.customers,customers.csv,1003
1,raw.accounts,accounts.csv,1253
2,raw.customer_accounts,customer_accounts.csv,1355
3,raw.transactions,transactions.csv,50010


## 9. Reconcile Source Rows Against Raw Tables

For every extract:

```text
source row count = raw table row count
```


In [9]:
def fetch_table_count(table_name: str) -> int:
    query = sql.SQL("SELECT COUNT(*) FROM raw.{}").format(
        sql.Identifier(table_name)
    )

    with get_connection() as conn:
        with conn.cursor() as cur:
            cur.execute(query)
            return cur.fetchone()[0]


reconciliation = []

source_counts_by_file = dict(
    zip(source_profile["file"], source_profile["rows"])
)

for table_name, spec in RAW_SOURCES.items():
    source_rows = int(source_counts_by_file[spec["file"].name])
    raw_rows = fetch_table_count(table_name)

    reconciliation.append(
        {
            "source_file": spec["file"].name,
            "raw_table": f"raw.{table_name}",
            "source_rows": source_rows,
            "raw_rows": raw_rows,
            "difference": raw_rows - source_rows,
            "reconciled": raw_rows == source_rows,
        }
    )

reconciliation = pd.DataFrame(reconciliation)
display(reconciliation)

if not reconciliation["reconciled"].all():
    raise RuntimeError("Raw ingestion reconciliation failed.")

print("All source files reconcile to their raw tables.")


,source_file,raw_table,source_rows,raw_rows,difference,reconciled
0,customers.csv,raw.customers,1003,1003,0,True
1,accounts.csv,raw.accounts,1253,1253,0,True
2,customer_accounts.csv,raw.customer_accounts,1355,1355,0,True
3,transactions.csv,raw.transactions,50010,50010,0,True


All source files reconcile to their raw tables.


## 10. Verify Ingestion Metadata

This confirms that raw rows retain both source-file lineage and ingestion time.


In [10]:
metadata_rows = []

with get_connection() as conn:
    with conn.cursor() as cur:
        for table_name in RAW_TABLES:
            query = sql.SQL(
                '''
                SELECT
                    _source_file,
                    COUNT(*) AS row_count,
                    MIN(_ingested_at) AS first_ingested_at,
                    MAX(_ingested_at) AS last_ingested_at
                FROM raw.{}
                GROUP BY _source_file
                ORDER BY _source_file;
                '''
            ).format(sql.Identifier(table_name))

            cur.execute(query)

            for source_file, row_count, first_ingested_at, last_ingested_at in cur.fetchall():
                metadata_rows.append(
                    {
                        "raw_table": f"raw.{table_name}",
                        "source_file": source_file,
                        "row_count": row_count,
                        "first_ingested_at": first_ingested_at,
                        "last_ingested_at": last_ingested_at,
                    }
                )

metadata_profile = pd.DataFrame(metadata_rows)
display(metadata_profile)


,raw_table,source_file,row_count,first_ingested_at,last_ingested_at
0,raw.customers,customers.csv,1003,2026-09-05 21:39:12.107338+00:00,2026-09-05 21:39:12.107338+00:00
1,raw.accounts,accounts.csv,1253,2026-09-05 21:39:12.176186+00:00,2026-09-05 21:39:12.176186+00:00
2,raw.customer_accounts,customer_accounts.csv,1355,2026-09-05 21:39:12.252896+00:00,2026-09-05 21:39:12.252896+00:00
3,raw.transactions,transactions.csv,50010,2026-09-05 21:39:12.321360+00:00,2026-09-05 21:39:12.321360+00:00


## 11. Inspect Raw Samples

The known malformed source records should still be visible in raw. That confirms we have not silently cleaned the source during ingestion.


In [11]:
def query_dataframe(query: str, params=None) -> pd.DataFrame:
    with get_connection() as conn:
        with conn.cursor() as cur:
            cur.execute(query, params)
            columns = [description.name for description in cur.description]
            rows = cur.fetchall()

    return pd.DataFrame(rows, columns=columns)


print("Sample raw customers")
display(
    query_dataframe(
        '''
        SELECT *
        FROM raw.customers
        ORDER BY customer_id
        LIMIT 5;
        '''
    )
)

print("\nKnown deliberately malformed transaction examples")
display(
    query_dataframe(
        '''
        SELECT
            transaction_id,
            account_id,
            transaction_timestamp,
            transaction_type,
            channel_code,
            amount,
            currency_code,
            status,
            _source_file,
            _ingested_at
        FROM raw.transactions
        WHERE transaction_id LIKE 'T_BAD_%'
           OR transaction_id = 'T_AFTER_CLOSE'
        ORDER BY transaction_id;
        '''
    )
)


Sample raw customers


,customer_id,customer_since_date,customer_status,_ingested_at,_source_file
0,C000001,2023-08-03,INACTIVE,2026-09-05 21:39:12.107338+00:00,customers.csv
1,C000002,2017-12-09,ACTIVE,2026-09-05 21:39:12.107338+00:00,customers.csv
2,C000003,2022-10-03,ACTIVE,2026-09-05 21:39:12.107338+00:00,customers.csv
3,C000004,2022-05-04,ACTIVE,2026-09-05 21:39:12.107338+00:00,customers.csv
4,C000005,2025-03-25,ACTIVE,2026-09-05 21:39:12.107338+00:00,customers.csv



Known deliberately malformed transaction examples


,transaction_id,account_id,transaction_timestamp,transaction_type,channel_code,amount,currency_code,status,_source_file,_ingested_at
0,T_AFTER_CLOSE,A000037,2025-08-16 00:00:00,PURCHASE,CARD,310.00,ZAR,SUCCESSFUL,transactions.csv,2026-09-05 21:39:12.321360+00:00
1,T_BAD_ACCOUNT,A999999,2026-08-20 10:15:00,PURCHASE,CARD,450.00,ZAR,SUCCESSFUL,transactions.csv,2026-09-05 21:39:12.321360+00:00
2,T_BAD_AMOUNT_NEG,A000004,2026-08-20 12:00:00,PURCHASE,CARD,-250.00,ZAR,SUCCESSFUL,transactions.csv,2026-09-05 21:39:12.321360+00:00
3,T_BAD_AMOUNT_TEXT,A000007,2026-08-20 14:00:00,PURCHASE,CARD,not_a_number,ZAR,SUCCESSFUL,transactions.csv,2026-09-05 21:39:12.321360+00:00
4,T_BAD_CHANNEL,A000003,2026-08-20 11:30:00,DEPOSIT,BRANCH,1500.00,ZAR,SUCCESSFUL,transactions.csv,2026-09-05 21:39:12.321360+00:00
5,T_BAD_CURRENCY,A000005,2026-08-20 12:30:00,TRANSFER,APP,800.00,RAND,SUCCESSFUL,transactions.csv,2026-09-05 21:39:12.321360+00:00
6,T_BAD_DATE,A000001,not-a-timestamp,PURCHASE,CARD,125.00,ZAR,SUCCESSFUL,transactions.csv,2026-09-05 21:39:12.321360+00:00
7,T_BAD_STATUS,A000006,2026-08-20 13:00:00,PURCHASE,CARD,220.00,ZAR,PENDING,transactions.csv,2026-09-05 21:39:12.321360+00:00
8,T_BAD_TYPE,A000002,2026-08-20 11:00:00,FEE,APP,35.00,ZAR,SUCCESSFUL,transactions.csv,2026-09-05 21:39:12.321360+00:00


## 12. Raw Ingestion Conclusion

The raw-ingestion step is complete when:

- PostgreSQL connection succeeds
- all four raw tables exist
- every source row is loaded
- source and database row counts reconcile
- `_source_file` is populated
- `_ingested_at` is populated
- intentionally invalid source values remain unchanged in raw

The database now contains:

```text
raw.customers
raw.accounts
raw.customer_accounts
raw.transactions
```

### Next step

`03_transform_staging.ipynb`

That notebook introduces the first real data-quality gate:

```text
raw
 │
 ├── valid rows   → staging
 │
 └── invalid rows → audit.rejected_records
```

It will also convert raw text into the PostgreSQL data types defined in the physical model.
